In [0]:
"""
03_fact_materials.py

Manufacturing Materials Fact Table

Business Grain:
    One material scan event.

Sources:
    material_events
    material_dimension
    supplier_dimension

Target:
    fact_materials

Author:
Sumanth Vempalle

Version:
2.3.0
"""

import dlt

from pyspark.sql.functions import col


# ============================================================
# Materials Fact Table
# ============================================================

@dlt.table(
    name="fact_materials",
    comment="Manufacturing Materials Fact Table.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def fact_materials():

    materials = dlt.read(
        "material_events"
    )

    material_dim = dlt.read(
        "material_dimension"
    )

    supplier_dim = dlt.read(
        "supplier_dimension"
    )

    return (

        materials.alias("m")

        .join(

            material_dim.alias("md"),

            on="material_number",

            how="left",

        )

        .join(

            supplier_dim.alias("sd"),

            on="supplier",

            how="left",

        )

        .select(

            # ====================================================
            # Event
            # ====================================================

            col("m.event_id"),

            col("m.event_timestamp"),

            col("m.event_version"),

            # ====================================================
            # Manufacturing Keys
            # ====================================================

            col("m.plant_code"),

            col("m.execution_id"),

            col("m.serial_number"),

            col("m.product_code"),

            # ====================================================
            # Material
            # ====================================================

            col("m.material_number"),

            col("m.batch_number"),

            col("md.product_name"),

            col("md.family"),

            # ====================================================
            # Supplier
            # ====================================================

            col("m.supplier"),

            # ====================================================
            # Measures
            # ====================================================

            col("m.scan_id"),

            col("m.scan_status"),

            # ====================================================
            # Audit
            # ====================================================

            col("m.silver_processing_timestamp"),

        )

    )